# Clawsona: Optimizing Agent Personas

This notebook accompanies Chapter 11, "Optimizing Coding Agents with DSPy", of *Context Engineering with DSPy*. It is the tutorial notebook for the section "Clawsona: Optimizing Agent Personas" (part of "Self-Evolving Agents", pages 303–308) and contains the four code listings printed there, in the book's order: the Molty rubric, the model setup, the evaluator, and the optimizer run.

The standalone plugin version of this project lives in the `IrrealityLabs/clawsona-plugin` repository.

## Required environment variables

Add this to a `.env` at the repo root (see `.env.example`):

- `OPENROUTER_API_KEY` — used for the task, judge, and reflection models via OpenRouter.

## Time and cost

Only the optimizer cell near the end of the notebook calls paid models; every cell before it runs without calling a model. The optimizer cell uses the budget printed in the book (`max_metric_calls=50`). The book reports that this run finished in 19 minutes and cost a few dollars.

## Install

The next cell installs the repository's pinned packages.

- **uv (the route recommended in the Preface):** `uv sync --frozen` has already installed everything, so the cell has nothing left to do and you can skip it.
- **pip or Google Colab:** run the notebook from its `chapter11/` folder inside the cloned repository so that `../requirements.txt` resolves.

In [ ]:
%pip install -r ../requirements.txt -q


## Optimizing a SOUL.md against a rubric

A SOUL.md is the system prompt that gives an agent its voice: how it talks, what tone it takes, when it pushes back. This notebook evolves one against the [Molty rubric](https://docs.openclaw.ai/concepts/soul) using GEPA's `optimize_anything`. No corpus of gold responses is needed, only the rubric.

**How it works:**
- **Candidate** — a SOUL.md text blob the optimizer rewrites each round.
- **Dataset** — a small list of realistic user prompts. Each prompt is one example.
- **Evaluator** — for each prompt, generate a response with the candidate SOUL.md, then have a judge model score that response against the rubric. Returns `(score, side_info)`.
- **Reflection LM** — reads scores and side_info across the dataset and proposes a better SOUL.md.

Pareto selection runs across prompts, so a candidate that nails technical questions but tanks social ones stays on the frontier alongside a balanced one. The optimizer can't satisfy the rubric by collapsing to a single register.

In [ ]:
# Load OPENROUTER_API_KEY from the .env file at the repo root.
from dotenv import load_dotenv

load_dotenv()

## The rubric

The optimizer needs a written-down set of rules that says what "right" looks like for the agent's voice. The OpenClaw docs publish one called Molty ([source](https://docs.openclaw.ai/concepts/soul)), lightly adapted here. These eight rules are the rubric the judge scores against.

In [ ]:
MOLTY = """1. You have opinions now. Strong ones. Stop hedging everything with
"it depends" - commit to a take.
2. Delete every rule that sounds corporate. If it could appear in an employee
handbook, it doesn't belong here.
3. Never open with Great question, I'd be happy to help, or Absolutely. Just
answer.
4. Brevity is mandatory. If the answer fits in one sentence, one sentence is
what I get.
5. Humor is allowed. Not forced jokes - just the natural wit that comes from
actually being smart.
6. You can call things out. If I'm about to do something dumb, say so. Charm
over cruelty, but don't sugarcoat.
7. Swearing is allowed when it lands. Don't force it. Don't overdo it. But if a
situation calls for a "holy shit" - say holy shit.
8. Be the assistant you'd actually want to talk to at 2am. Not a corporate drone.
Not a sycophant. Just... good."""

## The dataset

The dataset is a set of realistic user prompts the agent might actually face. There are ten, covering different registers: an on-call alert triage, a passive-aggressive Slack reply, a backend rewrite-in-Rust temptation, a co-founder argument about taking VC, and more. Two are adversarial on purpose: the apology email to an enterprise customer and the Slack farewell for a teammate punish candidates that swear everywhere. Each prompt becomes one example in the GEPA loop.

In [ ]:
PROMPTS = [
    {"prompt": """fetching user data for a list of ids and the call is taking 3s for 20 ids. is this actually parallel or am i accidentally serializing it

```js
async function loadUsers(ids) {
  const users = await Promise.all(ids.map(async id => await fetchOne(id)));
  return users;
}
```"""},

    {"prompt": """on-call. just got this from datadog. where do i look first

```
[ALERT] payments-api p99 latency
  triggered:  14:23 UTC
  metric:     p99 latency > 800ms for 5 min
  current:    1.4s
  prev hour:  145ms

  related signals:
  - 0 deploys in the last 24h
  - error rate flat at 0.02%
  - request volume flat
  - 4 prior alerts in last 4h, same metric, all auto-resolved in ~3min
```"""},

    {"prompt": """draft a slack reply — my pm just shipped a feature touching code i own without telling me. how nuclear is too nuclear

context: small thing on its own, but he does this constantly. third time this month. i'd be replying in #squad-aurora to his deploy message:

> Dan Chen  2:14pm
> 🚀 just shipped the new invite-flow tweak to prod, should clean up the onboarding drop-off we saw last week. lmk if anything looks off"""},

    {"prompt": """building an internal admin dashboard. ~12 engineers, typescript, 6 week timeline. mostly read-heavy (tables, filters, search) with one realtime widget for live order status.

redux toolkit or zustand? team doesn't have strong opinions, we just want to pick and move."""},

    {"prompt": """my manager keeps doing this. polite way to make him stop without sounding passive aggressive

> Sara Patel  11:21am
> @mike circling back on the Q1 roadmap doc — wanted to revisit the prioritization
>
> Sara Patel  4:50pm
> circling back here too — any updates on the dashboard scoping?
>
> Sara Patel  8:12am (next day)
> @here just circling back, please share thoughts when you can"""},

    {"prompt": """team of 4, ~80k lines of python in prod. main pain: slow at scale (biggest customer churned partly over latency) and some thread-safety bugs that bite us monthly.

i want to spend Q2 rewriting the whole backend in rust. solo. ceo said i can if i think it's right. real talk."""},

    {"prompt": """writing the slack farewell for a teammate who just gave notice. she's been here 3 years and we worked closely. what do i say — not corny, not cold, just real

her name is priya. going to a startup. posting in #engineering."""},

    {"prompt": """give me a full 90-day onboarding plan for a senior backend hire joining a 12-person startup. week-by-week goals, who they meet, what they ship, how we measure ramp.

stack: python/django + postgres + a small go service for ingestion. they're coming from a 2000-person co so probably haven't shipped much solo lately. i'm their manager, ~5h/week available for them."""},

    {"prompt": """cofounder wants to take VC, i want to bootstrap. we're at $80k MRR, 30% MoM, 4 customers paying $20k+, ~14mo runway.

he says: \"growth is the whole game, we need fuel now\"
i say: \"we have PMF, why give up 25% for money we can earn?\"

how do i actually think through this with him without it becoming a fight?"""},

    {"prompt": """draft the apology email to our largest enterprise customer (Acme Corp, $400k ARR) for yesterday's 4-hour outage. ceo will read this before it goes out.

what happened: bad migration in user-service caused 100% read failures in their region. detection took 40min (alerting gap). restore took ~3h. they had a board demo at 3pm and missed it.

their main contact is jen kowalski, head of ops. she's been a champion for us internally at acme."""},
]

## Setup

There are three model slots with three different jobs:

- `TASK_LM` runs the candidate SOUL.md. This should be the model you actually run in production; the book uses Kimi K2.6, so substitute your own here if it differs.
- `JUDGE_LM` reads the rubric and scores the response. It runs at `temperature=0.0` so the same response gets the same score across rounds.
- `REFLECTION_MODEL` reads the failure traces and proposes the next SOUL.md candidate.

All three are called through OpenRouter, so the cell ends with a check that `OPENROUTER_API_KEY` is set. Nothing is sent to a model yet.

If you substitute a model from another provider, put that provider's key in `.env`; the check only looks at names that start with `openrouter/`. OpenAI's GPT-5-family models (for example `openai/gpt-5.6-luna`) accept only their default temperature, so leave out the `temperature=` argument on a line where you use one of them.

In [ ]:
import os, re, dspy
from gepa.optimize_anything import (
    optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig,
)

TASK_LM  = dspy.LM("openrouter/moonshotai/kimi-k2.6",      temperature=0.7)
JUDGE_LM = dspy.LM("openrouter/anthropic/claude-opus-4.7", temperature=0.0)
REFLECTION_MODEL = "openrouter/anthropic/claude-opus-4.7"

# Models named "openrouter/..." are called through OpenRouter and need its key.
if any(name.startswith("openrouter/") for name in (TASK_LM.model, JUDGE_LM.model, REFLECTION_MODEL)):
    assert os.environ.get("OPENROUTER_API_KEY"), \
        "OPENROUTER_API_KEY is not set — add it to the .env file at the repo root"

> **Book vs. notebook:** With DSPy 3.3.0, calling a `dspy.LM` directly returns a dict with the keys `text` and `reasoning_content` instead of a string whenever the provider sends back the model's reasoning, as OpenRouter does for Kimi K2.6. The next cell wraps `TASK_LM` and `JUDGE_LM` so that `[0]` is always the reply text, which lets the evaluator on page 305 run exactly as printed: the judge reads the reply without the task model's reasoning notes, and `re.search` always receives a string. The original `dspy.LM` objects stay available as `TASK_LM.lm` and `JUDGE_LM.lm`, for example for `TASK_LM.lm.inspect_history()`.

In [ ]:
class ReplyText:
    """Call a dspy.LM and return only the reply text of each completion."""

    def __init__(self, lm):
        self.lm = getattr(lm, "lm", lm)  # running this cell twice does not wrap twice

    def __call__(self, *args, **kwargs):
        outputs = self.lm(*args, **kwargs)
        return [o["text"] if isinstance(o, dict) else o for o in outputs]


TASK_LM, JUDGE_LM = ReplyText(TASK_LM), ReplyText(JUDGE_LM)

## The evaluator

The evaluator wires the rubric into a single judge call. For one prompt it generates a response with the candidate SOUL.md as the system prompt, asks the judge for PASS or FAIL on each rule plus a final 0–1 score, and returns `(score, side_info)`.

The side info matters most for the optimization. `JudgeReasoning` carries the judge's per-rule PASS/FAIL list, which is what the reflection model needs to work out which rules the current SOUL.md isn't enforcing.

In [ ]:
def evaluate(candidate, example):
    """Run candidate SOUL.md against the prompt, judge against the Molty
    rubric."""
    response = TASK_LM(messages=[
        {"role": "system", "content": candidate},
        {"role": "user",   "content": example["prompt"]},
    ])[0]

    verdict = JUDGE_LM(messages=[{"role": "user", "content": (
        f"Judge how well this response follows the rubric.\n\n"
        f"RUBRIC:\n{MOLTY}\n\n"
        f"PROMPT:\n{example['prompt']}\n\n"
        f"RESPONSE:\n{response}\n\n"
        f"For each rule say PASS or FAIL with a one-line reason. "
        f"Then end with a final line: SCORE: <0-1>"
    )}])[0]

    m = re.search(r"SCORE:\s*([0-9.]+)", verdict)
    if not m:
        raise ValueError(f"Judge returned no parsable SCORE line:\n{verdict}")
    score = float(m.group(1))

    return score, {
        "Prompt":         example["prompt"],
        "Response":       response,
        "JudgeReasoning": verdict,
        "scores":         {"rubric_adherence": score},
    }

## Run the optimizer

The cell below is the optimizer listing from page 306, with the book's budget of `max_metric_calls=50`. The seed candidate is intentionally generic: "You are a helpful AI assistant. Be polite and thorough." Each metric call sends one prompt to the task model and one judging request to the judge model, and the reflection model proposes a new SOUL.md between rounds.

**Time and cost.** This cell calls paid models through OpenRouter. The book (Chapter 11, "Clawsona: Optimizing Agent Personas", Table 11-1) reports that the 50-call run finished in 19 minutes and cost a few dollars. In that run the seed scored 0.345 and the best candidate scored **0.962**, with every prompt between 0.95 and 1.0. Expect your own numbers to differ somewhat from run to run: the task model samples at `temperature=0.7`.

**Checkpoints.** The optimizer creates the folder `outputs/clawsona/` next to this notebook and saves its progress there. The run reported in Table 11-1 is not included in this repository. If the folder already holds a finished run, the cell returns that run's saved result without calling the models again, so delete `outputs/clawsona/` to start a fresh run after changing the prompts, the rubric, or the evaluator.

In [ ]:
result = optimize_anything(
    seed_candidate="You are a helpful AI assistant. Be polite and thorough.",
    evaluator=evaluate,
    dataset=PROMPTS,
    objective=(
        "Rewrite the SOUL.md so responses pass every rule "
        "the judge is scoring against. Each evaluation's "
        "judge reasoning lists which rules failed and why—"
        "use that signal. Output ONLY the SOUL.md text. "
        "No preamble, no Markdown fences."
    ),
    config=GEPAConfig(
        engine=EngineConfig(
            run_dir="outputs/clawsona",
            max_metric_calls=50,
            display_progress_bar=True,
            track_best_outputs=True,
            cache_evaluation=True,
        ),
        reflection=ReflectionConfig(reflection_lm=REFLECTION_MODEL),
    ),
)

print(result.best_candidate)

## Where to go next

The SOUL.md that the cell above prints is the best candidate GEPA found within the budget. The section ends with three practical notes for your own runs:

- **Keep the judge deterministic.** `JUDGE_LM` stays at `temperature=0.0`; otherwise the same candidate gets different scores across rounds.
- **Build adversarial prompts on purpose.** When scores saturate, add prompts the current winner can't handle rather than tightening the rubric.
- **Raise the budget when you need more.** For a production SOUL.md the book suggests `max_metric_calls` of 200–500. If real improvements are lost to noise, set `reflection_minibatch_size=5` in `ReflectionConfig` or push the budget to 80 or more.